In [1]:
import numpy as np
import pandas as pd

# Step 1. Calcualte EMA Crossover
### $EMA(P_t\frac{1}{n_{k,s}}) - EMA(P_t\frac{1}{n_{k,l}})$
- x > 0: long
- x < 0: short

In [2]:
EMA_PAIRS = [(8, 24), (16, 48), (32, 96)] # (n_ks, n_kl)

def ema(x, n):
    return x.ewm(alpha=1/n, adjust=False).mean() # Exponentially Weighted Moving

def ema_crossovers(price):
    signals = {}
    
    for k, (n_s, n_l) in enumerate(EMA_PAIRS, start=1):
        
        short = ema(price, n_s)
        long = ema(price, n_l)
        
        signals[f"x{k}"] = short - long
        
    return pd.DataFrame(signals)

# Step 2. First Volatility Normalization
### $y_{k,t} = \frac{x_{k,t}}{\sigma_{63}(P_t)}$
- 63 days for conventional markets: 3 months of market arctivity
- 91 for crypto market: market never closes


In [3]:
def first_norm(price, x, crypto=True):
    
    window = 91 if crypto else 63
    
    price_vol = price.rolling(window).std()
    
    return x.div(price_vol, axis=0)

# Step 3. Normalize Entire Signal
### $z_{k,t} = \frac{y_{k,t}}{\sigma(y_k)}$ 
- 252 days for conventional markets
- 365 days for crypto market

In [4]:
def second_norm(y, crypto=True):
    
    window = 365 if crypto else 252
    
    rolling_vol = y.rolling(window).std()
    
    return y / rolling_vol

# Step 4. Nonlinear Response Function
###  $u(z) =\frac{ze^{-\frac{z^2_k}{4} }}{\sqrt{2}e^{-\frac{1}{2}}}~~$    $~-1\le u \le 1$

In [ ]:
def u_func(z):
    den = np.sqrt(2) * np.exp(-0.5)
    
    return z * np.exp(-(z ** 2) / 4) / den

# Step 5. Create Combined Signal
### $\text{Signal}_t = \frac{1}{3}u_{1,t} + \frac{1}{3}u_{2,t} + \frac{1}{3}$

In [6]:
def momentum_signal(price, crytpo=True):
    
    xs = ema_crossovers(price)
    
    us = []
    
    for col in xs.columns:
        
        y = first_norm(price, xs[col], crypto=crytpo)
        
        z = second_norm(y, crypto=crytpo)
        
        u = u_func(z)
        
        us.append(u)
        
    U = pd.concat(us, axis=1)
    U.columns = ["u1", "u2", "u3"]
    
    return U.mean(axis=1)
    

# Portfolio Construction

## Strategy Returns with Signal Lag
### $\text{Signal}_{t-1}R_t$<br>

## Time-series Portfolio
### $w_{i,t}=\frac{\text{Signal}_{i,t}}{N}$

In [7]:
def ts_portfolio(signals, returns):
    
    n = signals.shape[1]
    
    weights = signals / n
    
    port_returns = (weights.shift(1) * returns).sum(axis=1)
    
    return port_returns, weights

## Cross-sectional Portfolio

In [8]:
def cs_weights(signals, n_l=3, n_s=3):
    
    weights = pd.DataFrame(0.0, index=signals.index, columns=signals.columns)
    
    total_positions = n_l + n_s
    position_size = 1 / total_positions
    
    for date, row in signals.iterrows():
        
        valid = row.dropna()
        if len(valid) < total_positions:
            continue
        
        l_assets = valid.nlargest(n_l).index
        s_assets = valid.nsmallets(n_s).index
        
        weights.loc[date, l_assets] = position_size
        weights.loc[date, s_assets] = -position_size
        
    return weights

def cs_portfolio(signals, returns):
    
    weights = cs_weights(signals)
    
    port_returns = (weights.shift(1) * returns).sum(axis=1)
    
    return port_returns, weights
    

# Rets and Metrics
## Rets

In [9]:
returns = prices.pct_change()

signals = pd.DataFrame(index=prices.index)

for ticker in prices.columns:
    
    signals[ticker] = momentum_signal(prices[ticker], crytpo=True)

NameError: name 'prices' is not defined

In [11]:
valid = (
    signals.notna().all(axis=1)
    & returns.notna().all(axis=1)
)

start = valid.idxmax()

signals = signals.loc[start:]
returns = returns.loc[start:]

NameError: name 'signals' is not defined

In [10]:
ts_rets, ts_weights = ts_portfolio(signals, returns)
cs_rets, cs_weights = cs_portfolio(signals, returns)

NameError: name 'signals' is not defined

## Metrics
1. Performance
2. Turnover